# grad-expressed-in-out — ex2: write tanh_back using cached out via (1 - out**2)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-expressed-in-out`. Running the final beacon cell reports progress against the `Backprop: grad expressed in out` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grad expressed in out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-expressed-in-out`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-expressed-in-out"
DD_SUBTOPIC = "Backprop: grad expressed in out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad expressed in `out` — quick refresher

Some elementwise back fns can be written purely in terms of the CACHED forward `out`, avoiding a second activation call:

```
sigmoid_back(grad_out, out, x) = grad_out * out * (1 - out)
tanh_back   (grad_out, out, x) = grad_out * (1 - out**2)
exp_back    (grad_out, out, x) = grad_out * out
```

The `(grad_out, out, x)` signature exists exactly so any of these back_fns can pick the cheapest cache. For `tanh`, the identity `d/dx tanh(x) = 1 - tanh(x)**2 = 1 - out**2` is the key — no second `t.tanh(x)` call, no `cosh`, no division. Just a square and a subtract.

### Exercise 2 — write tanh_back using cached out via (1 - out**2)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the 'grad expressed in out' pattern to a DIFFERENT activation by writing tanh_back as grad_out * (1 - out**2), reusing the cached forward output rather than recomputing tanh(x) or any cosh.
> Keywords: tanh, cached-out, elementwise, no-recompute
> ```

**KCs targeted:** `grad-expressed-in-out`, `back-fn-uses-cached-out`

Implement `tanh_back(grad_out, out, x)` using only the cached `out` — no `t.tanh(x)` recompute, no `cosh`.

**Math.** `out = tanh(x)`. The derivative collapses neatly:

```
d/dx tanh(x) = 1 - tanh(x)**2
             = 1 - out**2
```

So by the chain rule:

```
dL/dx = grad_out * (1 - out**2)
```

**Point of this drill.** It's the same SHAPE as `sigmoid_back` but a different activation — you're TRANSFERRING the cached-out pattern, not re-inventing it. One line should do it. The test feeds a deliberately WRONG `out` to catch any recompute-from-x.

**Inputs.** Plain `torch.Tensor`, same shape; float dtype. Output: tensor with the same shape as `x`.

In [ ]:
def tanh_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx tanh(x) = 1 - tanh(x)**2 = 1 - out**2.
    return grad_out * (1 - out**2)


<details><summary>Solution</summary>

```python
def tanh_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx tanh(x) = 1 - tanh(x)**2 = 1 - out**2.
    return grad_out * (1 - out**2)
```

**Same shape, different op.** Like sigmoid_back, every input to tanh_back is already cached — you just rearrange them. Watch the table of activations sharing this pattern: sigmoid uses `out * (1 - out)`; tanh uses `1 - out**2`; exp uses `out` directly. Each is a one-liner once you remember the identity.

**Why not `1 - t.tanh(x)**2`.** Bit-for-bit, with float math, `1 - t.tanh(x)**2 != 1 - out**2` if `out` and `t.tanh(x)` were computed with different rounding modes (rare, but possible on GPU). More importantly: a second tanh call is one more kernel launch per layer per backward — wasteful at scale.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()